# Digital Brain Experiment
**Can AI models replicate individual human brain organization?**

This notebook runs the full 5-level validation pipeline on the Algonauts 2023 dataset (NSD subset):
1. Encoding accuracy vs noise ceiling
2. Representational geometry preservation (RSA)
3. Cross-subject RSA identity matrix
4. Subject fingerprinting from digital brain predictions
5. Counterfactual consistency of subject differences

**Runtime:** ~45 min on Colab T4 GPU
**Data:** Algonauts 2023 (~4GB for 4 subjects)

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import os
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/digital-brain-results'
    os.makedirs(SAVE_DIR, exist_ok=True)
else:
    SAVE_DIR = './results'

print(f'Results will be saved to: {SAVE_DIR}')

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install -q osfclient tqdm transformers scikit-learn scipy matplotlib seaborn

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
CONFIG = {
    'subjects': [1, 2, 3, 4, 5, 6, 7, 8],   # All 8 Algonauts subjects
    'rois': [
        'V1v', 'V1d', 'V2v', 'V2d', 'V3v', 'V3d', 'hV4',
        'EBA', 'FBA-1', 'FBA-2',
        'OFA', 'FFA-1', 'FFA-2',
        'OPA', 'PPA', 'RSC',
        'OWFA', 'VWFA-1', 'VWFA-2',
        'early', 'midventral', 'midlateral', 'midparietal',
        'ventral', 'lateral', 'parietal',
    ],
    'clip_model': 'openai/clip-vit-large-patch14',
    'feature_batch_size': 256,
    'ridge_alphas': [0.01, 0.1, 1, 10, 100, 1000, 10000, 100000],
    'n_rsa_stimuli': 500,
    'n_permutations': 1000,
    'test_fraction': 0.15,
    'random_seed': 42,
}

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Subjects: {CONFIG["subjects"]}')

## Step 1: Download Algonauts 2023 Data

In [ ]:
import urllib.request
import zipfile
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR = Path('/content/algonauts_data') if IN_COLAB else Path('./data/algonauts')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Algonauts 2023 challenge data on OSF: https://osf.io/2rvsj/
# Per-subject download URLs (from the challenge documentation)
OSF_PROJECT = '2rvsj'

def download_subject_data(subj_id: int, data_dir: Path):
    """Download one subject's Algonauts 2023 data."""
    subj_str = f'subj{subj_id:02d}'
    subj_dir = data_dir / subj_str
    
    if (subj_dir / 'training_split').exists():
        print(f'  {subj_str}: already downloaded')
        return subj_dir
    
    subj_dir.mkdir(parents=True, exist_ok=True)
    
    # Try osfclient first
    try:
        import subprocess
        result = subprocess.run(
            ['python', '-m', 'osfclient', 'fetch', '-p', OSF_PROJECT,
             f'algonauts_2023_challenge_data/{subj_str}.zip',
             str(subj_dir / f'{subj_str}.zip')],
            capture_output=True, text=True, timeout=300
        )
        if result.returncode == 0:
            with zipfile.ZipFile(subj_dir / f'{subj_str}.zip', 'r') as z:
                z.extractall(subj_dir)
            (subj_dir / f'{subj_str}.zip').unlink()
            print(f'  {subj_str}: downloaded via osfclient')
            return subj_dir
    except Exception as e:
        print(f'  osfclient failed: {e}')
    
    # Manual fallback
    print(f'  Manual download required for {subj_str}.')
    print(f'  Visit: https://osf.io/2rvsj/ and download {subj_str} to {subj_dir}')
    return None


# Download all subjects
for sid in CONFIG['subjects']:
    download_subject_data(sid, DATA_DIR)

# Verify
available_subjects = [
    sid for sid in CONFIG['subjects']
    if (DATA_DIR / f'subj{sid:02d}' / 'training_split').exists()
]
print(f'\nAvailable subjects: {available_subjects}')

## Step 2: Load fMRI Data

In [ ]:
def load_subject_fmri(subj_id: int, hemisphere: str = 'lh') -> np.ndarray:
    """Load whole-brain fMRI betas. Returns (n_images, n_voxels)."""
    path = DATA_DIR / f'subj{subj_id:02d}' / 'training_split' / 'training_fmri' / f'{hemisphere}_training_fmri.npy'
    return np.load(path)

def load_roi_mask(subj_id: int, roi: str, hemisphere: str = 'lh') -> np.ndarray:
    """Load boolean ROI mask."""
    # Algonauts provides challenge_space ROI masks
    mask_dir = DATA_DIR / f'subj{subj_id:02d}' / 'roi_masks'
    # Try both naming conventions
    for pattern in [f'mapping_{roi}.npy', f'{hemisphere}_{roi}.npy', f'{roi}.npy']:
        path = mask_dir / pattern
        if path.exists():
            mask = np.load(path)
            return mask == 1  # some are stored as 0/1 int
    raise FileNotFoundError(f'ROI mask not found: {roi} for subj{subj_id:02d}')


# Load all subjects
print('Loading fMRI data...')
all_betas = {}   # {subj_id: (n_images, n_voxels)}
all_roi_betas = {}  # {subj_id: {roi: (n_images, n_voxels)}}

for sid in available_subjects:
    betas = load_subject_fmri(sid, hemisphere='lh')
    all_betas[sid] = betas
    
    roi_betas = {}
    for roi in CONFIG['rois']:
        try:
            mask = load_roi_mask(sid, roi)
            if mask.sum() > 0:
                roi_betas[roi] = betas[:, mask]
        except FileNotFoundError:
            pass
    
    all_roi_betas[sid] = roi_betas
    print(f'  subj{sid:02d}: {betas.shape}, {len(roi_betas)} ROIs')

n_total_images = list(all_betas.values())[0].shape[0] if all_betas else 0
print(f'\nTotal images per subject: {n_total_images}')

## Step 3: Load Stimuli and Extract CLIP Features

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor
from PIL import Image

@torch.no_grad()
def extract_clip_features(
    images: np.ndarray,  # (N, H, W, 3) uint8
    model_id: str = 'openai/clip-vit-large-patch14',
    batch_size: int = 256,
    device: str = 'cuda',
) -> np.ndarray:
    """Extract CLIP ViT-L/14 CLS token features. Returns (N, 1024)."""
    print(f'Loading {model_id}...')
    processor = CLIPProcessor.from_pretrained(model_id)
    model = CLIPModel.from_pretrained(model_id).to(device)
    model.eval()
    
    all_features = []
    for i in tqdm(range(0, len(images), batch_size), desc='Extracting CLIP features'):
        batch = [Image.fromarray(img) for img in images[i:i+batch_size]]
        inputs = processor(images=batch, return_tensors='pt', padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        out = model.vision_model(pixel_values=inputs['pixel_values'])
        feats = out.last_hidden_state[:, 0, :].cpu().numpy()
        all_features.append(feats)
    
    return np.concatenate(all_features, axis=0)


# Load stimuli (Algonauts provides training_images as .npy)
stimuli_path = DATA_DIR / f'subj{available_subjects[0]:02d}' / 'training_split' / 'training_images' / 'train_images.npy'
# Alternative: images may be stored per-subject or shared
if not stimuli_path.exists():
    # Look for shared stimulus set
    for candidate in DATA_DIR.rglob('*train*.npy'):
        if 'image' in candidate.name.lower():
            stimuli_path = candidate
            break

features_path = Path(SAVE_DIR) / 'clip_features_vit_l14.npy'

if features_path.exists():
    print('Loading cached features...')
    features = np.load(features_path)
elif stimuli_path.exists():
    print(f'Loading stimuli from {stimuli_path}...')
    stimuli = np.load(stimuli_path)
    print(f'Stimuli shape: {stimuli.shape}')
    features = extract_clip_features(stimuli, device=DEVICE)
    np.save(features_path, features)
    print(f'Features saved: {features.shape}')
else:
    raise FileNotFoundError(f'Stimuli not found at {stimuli_path}. Check data download.')

print(f'Features: {features.shape}  ({features.dtype})')

In [ ]:
# Train/test split — same indices for all subjects
rng = np.random.default_rng(CONFIG['random_seed'])
n_total = features.shape[0]
n_test = int(n_total * CONFIG['test_fraction'])
test_idx = rng.choice(n_total, size=n_test, replace=False)
train_idx = np.setdiff1d(np.arange(n_total), test_idx)

features_train = features[train_idx]
features_test = features[test_idx]

print(f'Train: {len(train_idx)} | Test: {len(test_idx)}')

## Step 4: Train Subject-Specific Digital Brains

In [ ]:
import pickle
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline


class DigitalBrain:
    """
    Subject-specific encoding model: maps stimuli → predicted brain responses.
    This IS the digital brain.
    """
    def __init__(self, subj_id, alphas=None):
        self.subj_id = subj_id
        self.alphas = alphas or CONFIG['ridge_alphas']
        self.roi_models = {}   # {roi: (feature_pipeline, voxel_pca)}
    
    def fit(self, features_train, roi_betas_train):
        """Train per-ROI ridge regression models."""
        for roi, betas in tqdm(roi_betas_train.items(), desc=f'subj{self.subj_id:02d}'):
            betas_tr = betas[train_idx]  # align to split
            n_vox = betas_tr.shape[1]
            n_components = min(100, n_vox, features_train.shape[0] - 1)
            
            # Reduce voxel dimensionality before fitting
            voxel_pca = PCA(n_components=n_components, random_state=42)
            betas_pcs = voxel_pca.fit_transform(betas_tr)
            
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('pca', PCA(n_components=min(500, features_train.shape[1]), random_state=42)),
                ('ridge', RidgeCV(alphas=self.alphas, cv=3, scoring='r2')),
            ])
            pipeline.fit(features_train, betas_pcs)
            self.roi_models[roi] = (pipeline, voxel_pca)
        return self
    
    def predict(self, features):
        """Predict brain responses. Returns {roi: (n_images, n_voxels)}."""
        preds = {}
        for roi, (pipeline, voxel_pca) in self.roi_models.items():
            pred_pcs = pipeline.predict(features)
            preds[roi] = voxel_pca.inverse_transform(pred_pcs)
        return preds
    
    def predict_roi(self, features, roi):
        pipeline, voxel_pca = self.roi_models[roi]
        return voxel_pca.inverse_transform(pipeline.predict(features))


# Train one digital brain per subject
digital_brains = {}
models_dir = Path(SAVE_DIR) / 'models'
models_dir.mkdir(exist_ok=True)

for sid in available_subjects:
    cache_path = models_dir / f'digital_brain_subj{sid:02d}.pkl'
    
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            brain = pickle.load(f)
        print(f'Loaded cached digital brain: subj{sid:02d}')
    else:
        brain = DigitalBrain(sid)
        brain.fit(features_train, all_roi_betas[sid])
        with open(cache_path, 'wb') as f:
            pickle.dump(brain, f)
    
    digital_brains[sid] = brain

print(f'\nTrained {len(digital_brains)} digital brains')
print(f'ROIs per brain: {len(list(digital_brains.values())[0].roi_models)}')

## Step 5: Generate Predictions

In [ ]:
print('Generating digital brain predictions on test set...')
predictions_test = {sid: brain.predict(features_test) for sid, brain in digital_brains.items()}

# Real betas on test set
real_test = {
    sid: {roi: betas[test_idx] for roi, betas in roi_betas.items()}
    for sid, roi_betas in all_roi_betas.items()
    if sid in digital_brains
}

# RSA subset (shared stimuli)
rsa_idx = test_idx[:CONFIG['n_rsa_stimuli']]
features_rsa = features[rsa_idx]
predictions_rsa = {sid: brain.predict(features_rsa) for sid, brain in digital_brains.items()}
real_rsa = {
    sid: {roi: betas[rsa_idx] for roi, betas in roi_betas.items()}
    for sid, roi_betas in all_roi_betas.items()
}

shared_rois = sorted(set.intersection(*[
    set(digital_brains[sid].roi_models.keys()) for sid in available_subjects
]))
print(f'Shared ROIs: {shared_rois}')

## Level 1: Encoding Accuracy

In [ ]:
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

def noise_ceiling_from_split_halves(betas: np.ndarray, n_splits: int = 20) -> float:
    """Estimate noise ceiling via split-half reliability."""
    rng_nc = np.random.default_rng(0)
    n = betas.shape[0]
    rs = []
    for _ in range(n_splits):
        idx = rng_nc.permutation(n)
        half1 = betas[idx[:n//2]].mean(axis=0)
        half2 = betas[idx[n//2:n//2*2]].mean(axis=0)
        r, _ = pearsonr(half1, half2)
        # Spearman-Brown correction
        nc = 2 * r / (1 + r)
        rs.append(nc)
    return float(np.mean(rs))

def encoding_r_per_roi(pred: np.ndarray, real: np.ndarray) -> float:
    """Median Pearson r across voxels."""
    min_v = min(pred.shape[1], real.shape[1])
    rs = [pearsonr(pred[:, v], real[:, v])[0] for v in range(min_v)]
    return float(np.nanmedian(rs))


print('Computing encoding accuracy...')
encoding_results = {}

for sid in available_subjects:
    encoding_results[sid] = {}
    for roi in shared_rois:
        pred = predictions_test[sid].get(roi)
        real = real_test[sid].get(roi)
        if pred is None or real is None:
            continue
        r = encoding_r_per_roi(pred, real)
        # Noise ceiling on ALL training betas for this ROI
        nc = noise_ceiling_from_split_halves(all_roi_betas[sid][roi]) if len(all_roi_betas[sid].get(roi, [])) > 0 else np.nan
        encoding_results[sid][roi] = {'median_r': r, 'noise_ceiling': nc, 'nc_ratio': r / max(nc, 1e-6)}

# Print summary
for sid in available_subjects:
    print(f'\nSubject {sid:02d}:')
    for roi in sorted(encoding_results[sid]):
        d = encoding_results[sid][roi]
        print(f'  {roi:<15} r={d["median_r"]:.3f}  NC={d["noise_ceiling"]:.3f}  NC_ratio={d["nc_ratio"]:.2f}')

## Level 2: Representational Geometry (RSA)

In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr

def compute_rdm(act: np.ndarray) -> np.ndarray:
    return squareform(pdist(act, metric='correlation'))

def rsa_score(rdm1, rdm2):
    n = rdm1.shape[0]
    idx = np.triu_indices(n, k=1)
    return spearmanr(rdm1[idx], rdm2[idx])[0]


# Per-ROI RSA: digital brain (self) vs real brain (self and others)
geometry_results = {}
rsa_idx_local = np.arange(len(rsa_idx))  # local indices into rsa arrays

for sid in available_subjects:
    geometry_results[sid] = {}
    others = [s for s in available_subjects if s != sid]
    
    for roi in shared_rois:
        pred = predictions_rsa[sid].get(roi)
        real_self = real_rsa[sid].get(roi)
        if pred is None or real_self is None:
            continue
        
        rdm_pred = compute_rdm(pred)
        rdm_self = compute_rdm(real_self)
        r_self = rsa_score(rdm_pred, rdm_self)
        
        r_others = []
        for oid in others:
            real_other = real_rsa[oid].get(roi)
            if real_other is not None:
                rdm_other = compute_rdm(real_other)
                r_others.append(rsa_score(rdm_pred, rdm_other))
        
        geometry_results[sid][roi] = {
            'rsa_self': r_self,
            'rsa_other_mean': float(np.mean(r_others)) if r_others else np.nan,
            'rsa_self_vs_other': r_self - float(np.mean(r_others)) if r_others else np.nan,
        }

# Summary table
print('RSA self vs. other (averaged across subjects):')
for roi in shared_rois:
    self_rs = [geometry_results[sid][roi]['rsa_self'] for sid in available_subjects if roi in geometry_results[sid]]
    other_rs = [geometry_results[sid][roi]['rsa_other_mean'] for sid in available_subjects if roi in geometry_results[sid]]
    if self_rs:
        print(f'  {roi:<15} RSA_self={np.mean(self_rs):.3f}±{np.std(self_rs):.3f}  '
              f'RSA_other={np.nanmean(other_rs):.3f}±{np.nanstd(other_rs):.3f}')

## Level 3: Cross-Subject RSA Identity Matrix

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

rsa_matrices = {}
n_subj = len(available_subjects)

for roi in shared_rois[:6]:  # compute for first 6 ROIs
    mat = np.zeros((n_subj, n_subj))
    for i, sid_pred in enumerate(available_subjects):
        pred = predictions_rsa[sid_pred].get(roi)
        if pred is None:
            continue
        rdm_pred = compute_rdm(pred)
        for j, sid_real in enumerate(available_subjects):
            real = real_rsa[sid_real].get(roi)
            if real is None:
                continue
            mat[i, j] = rsa_score(rdm_pred, compute_rdm(real))
    rsa_matrices[roi] = mat

# Diagonal dominance metric
print('RSA Identity Matrix — diagonal dominance:')
for roi, mat in rsa_matrices.items():
    dom = sum(1 for i in range(n_subj) if np.argmax(mat[i]) == i) / n_subj
    print(f'  {roi:<15} {dom:.0%} ({int(dom * n_subj)}/{n_subj} subjects self-identified)')

# Plot identity matrix for best ROI
best_roi = max(rsa_matrices, key=lambda r: sum(
    1 for i in range(n_subj) if np.argmax(rsa_matrices[r][i]) == i))
mat = rsa_matrices[best_roi]
labels = [f'S{sid:02d}' for sid in available_subjects]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(mat, cmap='RdYlGn', vmin=-0.1, vmax=0.8)
plt.colorbar(im, ax=ax, label='RSA (Spearman ρ)')
ax.set_xticks(range(n_subj)); ax.set_yticks(range(n_subj))
ax.set_xticklabels(labels); ax.set_yticklabels(labels)
ax.set_xlabel('Real brain (subject)'); ax.set_ylabel('Digital brain (subject)')
ax.set_title(f'Cross-Subject RSA Identity Matrix\nROI: {best_roi}', fontweight='bold')
for i in range(n_subj):
    ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor='blue', lw=2))
    for j in range(n_subj):
        ax.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center', fontsize=7)
fig.tight_layout()
fig.savefig(f'{SAVE_DIR}/rsa_identity_{best_roi}.pdf', bbox_inches='tight')
plt.show()
print(f'Saved to {SAVE_DIR}/rsa_identity_{best_roi}.pdf')

## Level 4: Subject Fingerprinting

In [ ]:
fingerprint_results = {}

for roi in shared_rois:
    # Template matching: which digital brain best predicts each subject's real responses?
    correct = 0
    for true_sid in available_subjects:
        real_resp = real_test[true_sid].get(roi)
        if real_resp is None:
            continue
        
        best_r = -np.inf
        best_sid = None
        for pred_sid in available_subjects:
            pred_resp = predictions_test[pred_sid].get(roi)
            if pred_resp is None:
                continue
            min_v = min(real_resp.shape[1], pred_resp.shape[1])
            r = np.corrcoef(real_resp[:, :min_v].ravel(), pred_resp[:, :min_v].ravel())[0, 1]
            if r > best_r:
                best_r = r
                best_sid = pred_sid
        
        if best_sid == true_sid:
            correct += 1
    
    n_s = len(available_subjects)
    acc = correct / n_s
    chance = 1.0 / n_s
    
    # Permutation test
    rng_fp = np.random.default_rng(42)
    perm_accs = []
    for _ in range(CONFIG['n_permutations']):
        perm_ids = rng_fp.permutation(available_subjects)
        perm_correct = sum(
            1 for true_sid, fake_sid in zip(available_subjects, perm_ids)
            if fake_sid == true_sid  # check if random assignment accidentally correct
        )
        perm_accs.append(perm_correct / n_s)
    
    # More rigorous: permutation on assignment
    perm_above = np.mean(np.array(perm_accs) >= acc)
    
    fingerprint_results[roi] = {'accuracy': acc, 'chance': chance, 'p': perm_above}
    sig = '***' if perm_above < 0.001 else '**' if perm_above < 0.01 else '*' if perm_above < 0.05 else 'ns'
    print(f'  {roi:<15} acc={acc:.1%}  chance={chance:.1%}  p={perm_above:.4f} {sig}')


# Figure
rois_to_plot = sorted(fingerprint_results.keys())
accs = [fingerprint_results[r]['accuracy'] for r in rois_to_plot]
chance_val = fingerprint_results[rois_to_plot[0]]['chance']
ps = [fingerprint_results[r]['p'] for r in rois_to_plot]

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(rois_to_plot))
colors = ['#2196F3' if p < 0.05 else '#9E9E9E' for p in ps]
ax.bar(x, accs, color=colors, alpha=0.85)
ax.axhline(chance_val, color='#F44336', linestyle='--', linewidth=1.5, label=f'Chance ({chance_val:.2f})')
for xi, (acc, p) in enumerate(zip(accs, ps)):
    stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    if stars:
        ax.text(xi, acc + 0.01, stars, ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(rois_to_plot, rotation=45, ha='right')
ax.set_ylabel('Identification Accuracy')
ax.set_title('Subject Fingerprinting: Digital Brain Identifies Real Subject', fontweight='bold')
ax.legend(); ax.set_ylim(0, 1.15)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(f'{SAVE_DIR}/fingerprinting.pdf', bbox_inches='tight')
plt.show()

## Level 5: Counterfactual Consistency

In [ ]:
cf_results = {}

for roi in shared_rois:
    pair_rs = []
    for a_idx, sid_a in enumerate(available_subjects):
        for sid_b in available_subjects[a_idx+1:]:
            real_a = real_test[sid_a].get(roi)
            real_b = real_test[sid_b].get(roi)
            pred_a = predictions_test[sid_a].get(roi)
            pred_b = predictions_test[sid_b].get(roi)
            if any(x is None for x in [real_a, real_b, pred_a, pred_b]):
                continue
            
            min_v = min(real_a.shape[1], real_b.shape[1], pred_a.shape[1], pred_b.shape[1])
            real_diff = (real_a[:, :min_v] - real_b[:, :min_v]).ravel()
            pred_diff = (pred_a[:, :min_v] - pred_b[:, :min_v]).ravel()
            
            r, _ = pearsonr(real_diff, pred_diff)
            pair_rs.append(r)
    
    if pair_rs:
        cf_results[roi] = {'mean_r': float(np.mean(pair_rs)), 'std_r': float(np.std(pair_rs)), 
                           'n_pairs': len(pair_rs)}
        print(f'  {roi:<15} mean_r={np.mean(pair_rs):.3f}±{np.std(pair_rs):.3f} (n_pairs={len(pair_rs)})')


# Violin plot
import seaborn as sns
fig, ax = plt.subplots(figsize=(14, 4))
rois_sorted = sorted(cf_results.keys())
means = [cf_results[r]['mean_r'] for r in rois_sorted]
ax.bar(range(len(rois_sorted)), means, color='#2196F3', alpha=0.8)
ax.axhline(0, color='red', linestyle='--', linewidth=1)
ax.set_xticks(range(len(rois_sorted)))
ax.set_xticklabels(rois_sorted, rotation=45, ha='right')
ax.set_ylabel('Counterfactual r (mean across pairs)')
ax.set_title('Counterfactual Consistency: Subject Differences Preserved in Digital Brain', fontweight='bold')
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
fig.savefig(f'{SAVE_DIR}/counterfactual.pdf', bbox_inches='tight')
plt.show()

## Summary Table

In [ ]:
import json

# Average encoding across subjects
summary = {}
for roi in shared_rois:
    enc_rs = [encoding_results[s][roi]['median_r'] for s in available_subjects if roi in encoding_results[s]]
    nc_ratios = [encoding_results[s][roi]['nc_ratio'] for s in available_subjects if roi in encoding_results[s]]
    self_rsas = [geometry_results[s][roi]['rsa_self'] for s in available_subjects if roi in geometry_results[s]]
    vs_others = [geometry_results[s][roi]['rsa_self_vs_other'] for s in available_subjects if roi in geometry_results[s]]
    fp_acc = fingerprint_results.get(roi, {}).get('accuracy', np.nan)
    cf_r = cf_results.get(roi, {}).get('mean_r', np.nan)
    
    summary[roi] = {
        'encoding_r_mean': float(np.nanmean(enc_rs)),
        'nc_ratio_mean': float(np.nanmean(nc_ratios)),
        'rsa_self_mean': float(np.nanmean(self_rsas)),
        'rsa_vs_other': float(np.nanmean(vs_others)),
        'fingerprint_acc': fp_acc,
        'counterfactual_r': cf_r,
    }

# Print and save
print(f'{'ROI':<15} {'Enc r':>7} {'NC ratio':>9} {'RSA self':>9} {'vs other':>9} {'FP acc':>7} {'CF r':>7}')
print('-' * 70)
for roi, d in sorted(summary.items()):
    def f(v): return f'{v:.3f}' if not np.isnan(v) else '  --  '
    print(f'{roi:<15} {f(d["encoding_r_mean"]):>7} {f(d["nc_ratio_mean"]):>9} '
          f'{f(d["rsa_self_mean"]):>9} {f(d["rsa_vs_other"]):>9} '
          f'{f(d["fingerprint_acc"]):>7} {f(d["counterfactual_r"]):>7}')

# Save all results
all_results = {
    'encoding': encoding_results,
    'geometry': geometry_results,
    'rsa_matrices': {roi: mat.tolist() for roi, mat in rsa_matrices.items()},
    'fingerprinting': fingerprint_results,
    'counterfactual': cf_results,
    'summary': summary,
}

def default(o):
    if isinstance(o, (np.integer, np.floating)): return o.item()
    raise TypeError

with open(f'{SAVE_DIR}/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=default)

print(f'\nAll results saved to {SAVE_DIR}/')